# Session 1: An Introduction to Deep Learning
## The Building Blocks of a Neural Network

In this notebook, we're going to put the fundamental concepts behind deep learning into practice.

Working through this notebook, we're going to:

1. Set up our Python environment.
2. Explicitly define a single neuron.
3. Stack neurons into a multilayer perceptron (MLP).
4. Test a simple network with and without activation functions.
5. Train a network on a simple task.
6. Split data into training, validation and test sets, and force a model to overfit.

Run the notebook one cell at a time and complete the activities to the best of your ability.

## 1. Setup

In this cell, we're going to load our libraries and fix a random seed.

As you may know, libraries contain pre-existing code, written by other developers, which we can use for our own applications. In this notebook, we'll use the following libraries:
- `random` — Python's standard pseudorandom number generator, included so we can set a fixed seed (see below).
- `numpy` (`np`) — A well-established library for working with numerical arrays (e.g. matrices, tensors) and vectorised maths.
- `matplotlib.pyplot` (`plt`) — Python's most common plotting library, used to plot figures.
- `torch` — The most popular Python package for deep learning, providing its own class of tensors and tensor operations, which are optimised to build neural networks.
- `torch.utils.data` — Specific functions from PyTorch for handling data loading. Here, we pull out `DataLoader`, which allows us to load data in batches, and `TensorDataset`, which bundles tensors together so they can be fed into `DataLoader`.

We'll also fix a random seed - this means that every time we run our code, we'll pick the same random numbers. This makes our code fully reproducible and is great for debugging.

In [ ]:
# See above for a description of these libraries
import random
import numpy as np
import matplotlib.pyplot as plt

# PyTorch (written here as torch) is a library specifically designed for building neural networks
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Fix a random seed
SEED = 42
# Different libraries have different random number generators, we need to set the seed for all of them
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
# If a CUDA-compatible GPU is available, we set the seed for CUDA too
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# Force the GPU to use deterministic algorithms so results stay reproducible
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Set the device - this tells PyTorch which part of your computer to use to train the model (e.g. GPU/CUDA or CPU).
# If you have GPU, you should see cuda here (though our model is small enough to run on CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. A Single Neuron

Almost everything in deep learning is built upon **neurons**. A neuron takes some numbers in, multiplies each one by a **weight**, adds them together with a **bias**, and feeds the result through an **activation function**. The output of a neuron is given by the following formula:

$$
y = f\left(\sum_i \left(x_i \times w_i\right) + b\right)
$$

where $x_i$ are the inputs, $w_i$ are the channel weights, $b$ is the neuron's bias and $f$ is the activation function. 

Below, we write that equation out in code and run the example from the lecture notes: weights $w_1 = 0.5$, $w_2 = 1.0$, bias $b = 1$, and inputs $x_1 = 2$, $x_2 = 3$.

**Activity:** Change `w` and `b` in the cell below and re-run it. Can you find values which give an output of exactly 6?

In [ ]:
# Define a single neuron
def neuron(inputs, weights, bias, activation):
    # Multiply every input by its own channel weight, sum the results, then add the neuron's bias
    total = (np.asarray(inputs) * np.asarray(weights)).sum() + bias
    # Feed the total through the activation function to get the neuron's output
    return activation(total)

# The identity activation function - returns whatever it was given
def identity(x):
    return x

# Inputs
x = [2.0, 3.0]
# Weights
w = [0.5, 1.0]
# Bias
b = 1.0
# Run the neuron and print the result
print(f"Neuron output: {neuron(x, w, b, identity)}")

## 3. From Neurons to an MLP

In practice we stack neurons into **layers**. A stack of layers is called a **multilayer perceptron** (MLP). An MLP has the following layers:
- The **input layer** - takes in the data.
- The **hidden layers** - learn intermediate representations.
- The **output layer** - produces the final answer.

In a **fully-connected** layer, every neuron in one layer connects to every neuron in the next.

Instead of defining a layer ourselves, we'll use PyTorch. PyTorch allows us to define different layer types and manipulate their shapes. One way to do this is by using an `nn.Sequential` container, which we can populate with a sequence of layers (in order). 

**Activity:** Add another hidden layer to the MLP below, or make the existing ones wider. Change the output layer dimension and re-run the cell.

In [ ]:
# Building a small MLP
# We can use the nn.Sequential container from PyTorch to apply each layer in the order we list them
# Each call to nn.Linear adds a new, fully-connected layer to the network
# We can also add an activation function after each layer
mlp = nn.Sequential(
    # Input layer to first hidden layer (3 inputs -> 5 neurons)
    nn.Linear(3, 5),
    # Activation function, applied to every neuron in the hidden layer
    nn.ReLU(),
    # First hidden layer to second hidden layer (5 inputs -> 5 neurons)
    nn.Linear(5, 5),
    # Another activation function
    nn.ReLU(),
    # Second hidden layer to output layer (5 inputs -> 2 neurons)
    nn.Linear(5, 2),
)

# Push a batch of five examples through the network, each with three input features
out = mlp(torch.randn(5, 3))
print(out)
# The batch size stays the same, but the number of features changes to match the output layer
print(f"\nInput batch shape: (5, 3) -> output shape: {tuple(out.shape)}")
# The network isn't trained at all yet, so the output is random

In [ ]:
# We can plot a schematic of the network architecture with the following code:
fig, ax = plt.subplots(figsize=(7, 4))
# The architecture we built above, written as a list of layer sizes
linears = [m for m in mlp if isinstance(m, nn.Linear)]
layer_sizes = [linears[0].in_features] + [m.out_features for m in linears]
# Work out the (x, y) position of every neuron, centring each layer vertically on zero
positions = [[(i, y) for y in np.linspace(-(n - 1) / 2, (n - 1) / 2, n)] for i, n in enumerate(layer_sizes)]
# Draw one faint grey line per channel - every neuron connects to every neuron in the next layer
for left, right in zip(positions[:-1], positions[1:]):
    for (x0, y0) in left:
        for (x1, y1) in right:
            ax.plot([x0, x1], [y0, y1], color="gray", linewidth=0.7, alpha=0.6, zorder=1)
# Colour the input layer, the hidden layers and the output layer differently
colours = ["steelblue"] + ["seagreen"] * (len(layer_sizes) - 2) + ["darkorange"]
# Name each layer so we can label it underneath the diagram
names = ["input"] + [f"hidden {i + 1}" for i in range(len(layer_sizes) - 2)] + ["output"]
# Draw the neurons themselves on top of the channels
for i, layer in enumerate(positions):
    # Each neuron is a large circle, coloured by which layer it belongs to
    ax.scatter([p[0] for p in layer], [p[1] for p in layer], s=320, color=colours[i], edgecolors="black", zorder=2)
    # Label the layer underneath the diagram
    ax.text(i, -3.0, f"{names[i]}\n({layer_sizes[i]})", ha="center", fontsize=9)
# Leave enough room below the diagram for those labels
ax.set_ylim(-3.8, 2.5)
# Hide the axes
ax.axis("off")
# Set the figure title, tidy spacing and show
ax.set_title("A multilayer perceptron")
plt.tight_layout()
plt.show()

## 4. Activation Functions

Activation functions introduce **non-linearity**, which is what lets a network learn complex relationships. Let's check that claim numerically.

In [ ]:
# Fix the seed again so we get the same random numbers
torch.manual_seed(SEED)

# Without activation functions
# Let's define a simple MLP - the same as above but with input and output size of 1 and no activation functions
simple_mlp = nn.Sequential(
    nn.Linear(1, 5),
    nn.Linear(5, 5),
    nn.Linear(5, 1),
)

# Let's try inputting 10 evenly-spaced values
x_vals = torch.linspace(-3, 3, 10)
# Empty list to store each output
y_vals = []
# Loop over the ten inputs one at a time
for x in x_vals:
    # Input the value through the network and get the output
    y = simple_mlp(x.unsqueeze(0))
    y_vals.append(y.item())

# Plot the ten input-output pairs
plt.plot(x_vals.numpy(), y_vals, marker="o")
# Label the axes
plt.xlabel("input")
plt.ylabel("output")
# Set the figure title
plt.title("A stack of linear layers gives a straight line")
# Show the figure
plt.show()

In [ ]:
# Fix the seed again so we get the same random numbers
torch.manual_seed(SEED)

# With activation functions
# Now let's do the same thing, but using a ReLU activation function
simple_mlp = nn.Sequential(
    nn.Linear(1, 5),
    nn.ReLU(),
    nn.Linear(5, 5),
    nn.ReLU(),
    nn.Linear(5, 1),
)

# Let's try inputting 10 evenly-spaced values
x_vals = torch.linspace(-3, 3, 10)
# Empty list to store each output
y_vals = []
# Loop over the ten inputs one at a time
for x in x_vals:
    # Input the value through the network and get the output
    y = simple_mlp(x.unsqueeze(0))
    y_vals.append(y.item())

# Plot the ten input-output pairs
plt.plot(x_vals.numpy(), y_vals, marker="o")
# Label the axes
plt.xlabel("input")
plt.ylabel("output")
# Set the figure title
plt.title("Activation functions introduce non-linearity")
# Show the figure
plt.show()

## 5. How Neural Networks Learn

Training means finding the weights and biases that make the model's output as close as possible to what we want. Training is broken down into the following steps:

1. **Forward pass** - Pass the input through the network to get an output.
2. **Calculate Loss** - Measure how wrong that output is. A loss of 0 means a perfect answer.
3. **Backpropagation** - Work out the gradient of the loss with respect to every parameter, i.e. which way to nudge each parameter to make the loss go down.
4. **Gradient descent** - Update the parameters in the direction that should make the loss go down. The **learning rate** controls how big each update is.
5. **Repeat**.

We're going to train a very simple network, comprised of a single neuron, to multiply every input by 2. Our training data is the set of pairs $(x = 1, y = 2)$, $(x = 2, y = 4)$, and so on.

PyTorch's **autograd** system tracks every operation we perform, so when we call `loss.backward()` it works out the gradients for us. We'll use the **Adam** optimiser, which adapts the step size for each parameter individually, and **mean squared error** as our loss.

We also record model performance on the validation set, which isn't explicitly used in training. This tells us if our model is actually learning to generalise to unseen data, or just overfitting to the training data. To do this, we switch off PyTorch's built in "autograd" system. Autograd calculates gradients automatically as we perform tensor operations, which makes training much faster. We don't need gradients for validation (since we never train the model on validation data), so we can turn it off.

**Activity:** Replace the simple, single-neuron network below with a more complex network (like the one in the cell above). What happens to the outputs?

In [ ]:
# Training data for our "multiply by two" task - five paired inputs (x) and expected outputs (y)
x_train = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0]).unsqueeze(1)
y_train = 2.0 * x_train
# Held-out validation pairs - allows us to check how well the model performs on unseen data
x_val = torch.tensor([0.5, 1.5, 6.0, 7.0]).unsqueeze(1)
y_val = 2.0 * x_val

# Define a single neuron with one input and one output
network = nn.Linear(1, 1)
# Set our optimiser to Adam
optimiser = torch.optim.Adam(network.parameters(), lr=0.1)
# Initialise empty lists to store the training and validation loss at every step
train_losses = []
val_losses = []

# Let's see what the untrained network predicts (this should be random)
# Evenly-spaced inputs so we can plot the neuron's input-output line
x_vals = torch.linspace(-6, 6, 20).unsqueeze(1)
# Using torch.no_grad() means we don't compute gradients for this operation (we're not training here)
with torch.no_grad():
    y_before = network(x_vals)

# Now let's train the neuron
# First we define how many epochs we'll train for
epochs = 500
# Then we iterate over each epoch
for step in range(epochs):
    # Forward pass - put the inputs through the network to get predictions
    y_pred = network(x_train)
    # Calculate the loss (mean squared error between prediction and target)
    loss = F.mse_loss(y_pred, y_train)
    # Reset the gradients left over from the previous step
    optimiser.zero_grad()
    # Backpropagate to work out the gradient of the loss for every parameter
    loss.backward()
    # Update the parameters using those gradients
    optimiser.step()
    # Record this step's training loss
    train_losses.append(loss.item())
    # Measure the loss on the validation set
    with torch.no_grad():
        val_losses.append(F.mse_loss(network(x_val), y_val).item())

# Check what the neuron actually learned
linears = [m for m in network.modules() if isinstance(m, nn.Linear)]
if len(linears) == 1 and linears[0].in_features == 1 and linears[0].out_features == 1:
    print(f"\nLearned weight: {linears[0].weight.item():.4f} (we were hoping for 2.0)")
    print(f"Learned bias:   {linears[0].bias.item():.4f} (we were hoping for 0.0)")

# Record the trained outputs on the same evenly-spaced inputs
with torch.no_grad():
    y_after = network(x_vals)

# Plot the untrained and trained input-output lines
plt.plot(x_vals.numpy(), y_before.numpy(), marker="o", label="before training")
plt.plot(x_vals.numpy(), y_after.numpy(), marker="o", label="after training")
# The true relationship we were hoping to learn
plt.plot(x_vals.numpy(), (2.0 * x_vals).numpy(), linestyle="--", label="target (y = 2x)")
# Label the axes
plt.xlabel("input")
plt.ylabel("output")
# Set the figure title
plt.title("A neuron learning to multiply by two")
plt.legend()
# Show the figure
plt.show()


### Let's plot the losses

In [ ]:
# Plot the training and validation losses we recorded above
plt.plot(train_losses, color="steelblue", label="training loss")
plt.plot(val_losses, color="darkorange", label="validation loss")
# The losses fall a long way, so use a log scale
plt.yscale("log")
# Label the axes
plt.xlabel("training step")
plt.ylabel("MSE loss")
# Set the figure title
plt.title("Training and validation loss")
plt.legend()
# Show the figure
plt.show()


## 6. Train, Validation and Test Sets

A network that fits its training data hasn't necessarily learned anything useful, a big enough model can simply memorise every training example and fail on new data. This is what we call **overfitting**. To make sure our models aren't overfitting, we split the data three ways:

- The **training set** - The data the model learns from.
- The **validation set** - Passed through the model at the end of every epoch to check whether it's improving on unseen data (**never** used to update the parameters).
- The **test set** - Kept unseen until the very end, giving the closest estimate of unbiased performance on truly unseen data.

As you might recall, a **batch** is a small group of training examples processed together, and one full pass through all the batches is an **epoch**. PyTorch's `DataLoader` handles the batching for us.

Below we fit a deliberately over-complicated network to 60 noisy measurements and train it for much longer than necessary. Our aim is to re-create a sine distribution.

Let's start by defining our dataset - 100 pairs of points on a sine distribution with noise.

In [ ]:
# Let's build a small, noisy dataset
rng = np.random.default_rng(SEED)
# Define the number of measurements we have in total
N_POINTS = 100
# Draw the input values at random
x_all = rng.uniform(-3, 3, N_POINTS)
# Define the underlying relationship we want the network to discover, plus measurement noise
y_all = np.sin(x_all) + rng.normal(0, 0.3, N_POINTS)

# Shuffle the data before splitting
perm = rng.permutation(N_POINTS)
# Split the shuffled data 60% train, 20% validation, 20% test
train_idx, val_idx, test_idx = np.split(perm, [int(0.6 * N_POINTS), int(0.8 * N_POINTS)])

# Define a helper function which turns a set of indices into a pair of PyTorch tensors
def make_split(idx):
    return (torch.tensor(x_all[idx], dtype=torch.float32).unsqueeze(1),
            torch.tensor(y_all[idx], dtype=torch.float32).unsqueeze(1))

# Build the three splits
x_tr, y_tr = make_split(train_idx)
x_va, y_va = make_split(val_idx)
x_te, y_te = make_split(test_idx)
print(f"Train: {len(x_tr)} points | Validation: {len(x_va)} points | Test: {len(x_te)} points")

# The DataLoader bundles inputs and targets together and hands them back in shuffled batches.
# Our training set is small enough to fit in a single batch - in later sessions we'll use thousands of images
train_loader = DataLoader(TensorDataset(x_tr, y_tr), batch_size=60, shuffle=True)

### Now let's create our network

In [ ]:
# Our network contains 256 neurons per hidden layer (which is overkill for only 60 training points)
network = nn.Sequential(
    nn.Linear(1, 256),
    nn.ReLU(),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Linear(256, 1)
)
# Set up the optimiser
optimiser = torch.optim.Adam(network.parameters(), lr=0.005)
# Record the training and validation loss at every epoch
train_losses = []
val_losses = []
# Track the lowest validation loss we've seen, and the parameters that produced it
best_val, best_epoch, best_state = float("inf"), 0, None
# Train for a very long time
EPOCHS = 10000


# Now we train
for epoch in range(1, EPOCHS + 1):
    # Iterate over the batches the DataLoader gives us
    for xb, yb in train_loader:
        # Forward pass, then calculate the loss for this batch
        loss = F.mse_loss(network(xb), yb)
        # Reset gradients, backpropagate, and update the parameters
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    # Disable autograd
    with torch.no_grad():
        val_loss = F.mse_loss(network(x_va), y_va).item()
    # Record both losses for this epoch
    train_losses.append(loss.item())
    val_losses.append(val_loss)
    # If this is the best validation loss so far, keep a copy of the parameters that produced it
    if val_loss < best_val:
        best_val, best_epoch = val_loss, epoch
        best_state = {k: v.clone() for k, v in network.state_dict().items()}

# Report where the validation loss was at its minimum, and where it ended up
print(f"Best validation loss {best_val:.4f} at epoch {best_epoch}, final validation loss {val_losses[-1]:.4f}")


# Here is where we use the test set - we'll check performance on the final epoch and the epoch with lowest validation loss
# We'll also record model performance on a dense range of inputs (which we use in the next cell)
x_dense = torch.linspace(-3, 3, 300).unsqueeze(1)
# Get the final epoch's predictions and its loss on the test set
with torch.no_grad():
    test_final = F.mse_loss(network(x_te), y_te).item()
    pred_final = network(x_dense).squeeze(1).numpy()
# Now load the parameters from the best epoch back into the network and repeat
network.load_state_dict(best_state)
with torch.no_grad():
    test_best = F.mse_loss(network(x_te), y_te).item()
    pred_best = network(x_dense).squeeze(1).numpy()

# Print the loss on the test set for both
print(f"Test MSE using the final epoch: {test_final:.4f} | using the best-validation epoch: {test_best:.4f}")

### Plotting the results

We start by plotting the loss curves (left). The training loss keeps falling as the model memorises the noise, while the validation loss starts to rise.

We can see the outputs of the overfitted model (right) bends towards individual noisy training points, while the model saved at the best validation epoch stays closer to the true relationship.

In [ ]:
# This code just plots our losses and model outputs:
# Set up two panels side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
# Plot the training loss
axes[0].plot(train_losses, color="steelblue", linewidth=2, label="training loss")
# Plot the validation loss
axes[0].plot(val_losses, color="darkorange", linewidth=2, label="validation loss")
# Mark the epoch where the validation loss was lowest
axes[0].axvline(best_epoch, color="green", linestyle="--", linewidth=1.5, label=f"best epoch ({best_epoch})")
# Use log scales
axes[0].set_xscale("log")
axes[0].set_yscale("log")
# Label the axes and set the panel title
axes[0].set_xlabel("epoch (log scale)")
axes[0].set_ylabel("MSE loss (log scale)")
axes[0].set_title("Training loss falls, validation loss turns back up")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)
# Plot the true underlying relationship
axes[1].plot(x_dense.squeeze(1).numpy(), np.sin(1.5 * x_dense.squeeze(1).numpy()) + 0.3 * x_dense.squeeze(1).numpy(),
             color="black", linestyle="--", linewidth=2, label="true relationship")
# Plot the training points
axes[1].scatter(x_all[train_idx], y_all[train_idx], s=35, color="lightgray", edgecolors="gray",
                label="training data", zorder=2)
# Plot the overfitted model (the one from the final epoch)
axes[1].plot(x_dense.squeeze(1).numpy(), pred_final, color="crimson", linewidth=2,
             label=f"final epoch (test MSE {test_final:.2f})")
# Plot the model from the epoch with the lowest validation loss
axes[1].plot(x_dense.squeeze(1).numpy(), pred_best, color="seagreen", linewidth=2,
             label=f"best validation epoch (test MSE {test_best:.2f})")
# Label the axes and set the panel title
axes[1].set_xlabel("input x")
axes[1].set_ylabel("output y")
axes[1].set_ylim(-2.5, 2.5)
axes[1].set_title("Memorising the noise vs learning the relationship")
axes[1].legend(fontsize=8)
# Tidy spacing and show the figure
plt.tight_layout()
plt.show()